# Session 14 — From a dense model to a mixture of experts

**The brief.** Train a dense ("linear") model and convert it into an MoE. Model size and data
are our call, but the MoE must be shown to keep training and keep reducing the loss.

**The decisions behind this notebook** are recorded, with their reasons, in
`assignment_plan.html`. In one line each:

| decision | choice |
|---|---|
| model | session 13's 16 layers × width 320, 8K-token tokenizer (22.4M parameters) |
| data | FineWeb sample-10BT; R1 reads the first 50M tokens, R2 and R3 the next 50M |
| conversion | option E: each FFN cut into 4 slices of 320, each copied 8× → 32 experts in 4 families, one per family per token |
| clones | probabilistic selection inside each family for the first ~300 steps |
| control | the dense model continues for the same 50M tokens (R2) |
| schedule | warmup-stable-decay: R1 never decays, R2 and R3 decay in their last 20% |
| fixed | Colab T4, fp16, sequence 512, batch 32, lr 1e-3 |

### This notebook grows in three versions

```
v1  (this one)  data, the dense model, run R1 ............ you run it, we look at it
v2              + the MoE layer, the converter, the gates . run on a small slice
v3              + runs R2 and R3, analysis and figures .... the full comparison
```

Each version only **adds cells at the end**. The finished R1 is cached on your Drive, so v2
and v3 load it instead of training it again. For that reason the dense model and the training
loop (Cells 5 and 6) are frozen after v1.

### Running this

Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. Everything needed for the
write-up lands in `outputs/` on your Drive and is zipped by the last cell. If Colab
disconnects, run all again: finished stages reload from the cache, and an interrupted run
resumes from its last checkpoint.

## Cell 1 — environment

### What this block does
Imports, fixes the seed, and decides the **profile**: `full` on a GPU (the real experiment),
`smoke` on a GPU (real model and data, a few million tokens, to test every code path), or
`cpu`, a tiny version of everything used to test the notebook on a laptop.

### What you should see
A version box, the GPU name, and `profile: full`. If it says `cpu`, you are not on a GPU
runtime and none of the numbers are the real experiment.

In [1]:
import sys, subprocess

def _pip(mod, pkg=None):
    """Install only if the import actually fails - Colab already has most of this."""
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg or mod], check=False)

_pip("tokenizers")
_pip("datasets")

import os, math, time, json, zipfile, random, platform, hashlib, inspect, io
from contextlib import redirect_stdout, nullcontext
from dataclasses import dataclass, asdict, replace

try:
    sys.stdout.reconfigure(encoding="utf-8")   # Windows consoles default to cp1252
except Exception:
    pass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

NOTEBOOK_VERSION = "v1"
SEED = 1337
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

# ┌──────────────────────────────────────────────────────────────────────────────────────┐
# │ MODE: "full" is the experiment. "smoke" runs the real model on ~2M tokens per run to   │
# │ check every GPU code path in a few minutes (plus the one-time data preparation).       │
# └──────────────────────────────────────────────────────────────────────────────────────┘
MODE = "full"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROFILE = os.environ.get("ERA5_PROFILE") or (MODE if DEVICE == "cuda" else "cpu")
AMP = DEVICE == "cuda"                  # fp16 autocast + loss scaling (a T4 has no bf16)
AMP_DTYPE = torch.float16

if DEVICE == "cuda":
    _p = torch.cuda.get_device_properties(0)
    GPU_NAME, VRAM_GB = _p.name, _p.total_memory / 1e9
else:
    GPU_NAME, VRAM_GB = None, 0.0

_BANNER = [f"NOTEBOOK {NOTEBOOK_VERSION}  -  Session 14, dense model to mixture of experts",
           f"profile: {PROFILE}   device: {DEVICE}" + (f"   gpu: {GPU_NAME}" if GPU_NAME else "")]
if PROFILE == "cpu":
    _BANNER += ["", "!! REDUCED PROFILE: tiny model, tiny data.",
                "!! Use it to test the notebook. None of these numbers are the experiment."]
elif PROFILE == "smoke":
    _BANNER += ["", "SMOKE TEST: real model and data, ~2M tokens per run.",
                "Checks every GPU code path. Set MODE = \"full\" afterwards for the experiment."]
_w = max(len(x) for x in _BANNER) + 4
print("+" + "-" * _w + "+")
for _line in _BANNER:
    print("|  " + _line.ljust(_w - 2) + "|")
print("+" + "-" * _w + "+\n")
print(f"python : {platform.python_version()}")
print(f"torch  : {torch.__version__}")
if GPU_NAME:
    print(f"gpu    : {GPU_NAME}, {VRAM_GB:.1f} GB")

R = {"meta": {"notebook_version": NOTEBOOK_VERSION, "torch": torch.__version__,
              "python": platform.python_version(), "seed": SEED, "device": DEVICE,
              "profile": PROFILE, "gpu": GPU_NAME, "vram_gb": VRAM_GB,
              "started": time.strftime("%Y-%m-%d %H:%M:%S")},
     "gates": {}}

+-----------------------------------------------------------------+
|  NOTEBOOK v1  -  Session 14, dense model to mixture of experts  |
|  profile: full   device: cuda   gpu: Tesla T4                   |
+-----------------------------------------------------------------+

python : 3.13.15
torch  : 2.11.0+cu128
gpu    : Tesla T4, 15.6 GB


## Cell 2 — configuration

### What this block does
Every number that shapes the experiment, in one place, printed.

### Why these values
The model and the optimiser settings are session 13's, which you approved reusing ("similar
to last assignment"). Two things are new:

- **The schedule is warmup-stable-decay (WSD).** The learning rate rises for about 200
  steps, then stays flat. It is only lowered in the last 20% of R2 and R3. R1 never decays,
  so the conversion in v2 happens while the model can still learn quickly. Session 13's cosine
  schedule would have shrunk the learning rate to 10% by then.
- **Twice the data.** R1 reads the first 50M tokens. R2 and R3 continue on the *next* 50M,
  so the continuations never see a repeated sentence.

In [2]:
@dataclass(frozen=True)
class ModelCfg:
    vocab: int = 8192
    d: int = 320            # width: how many numbers carry one token between layers
    n_layer: int = 16       # depth
    n_head: int = 5         # 320 = 5 heads x 64
    seq: int = 512
    mlp_mult: int = 4       # feed-forward middle = 4 x width = 1,280 neurons

@dataclass(frozen=True)
class TrainCfg:
    tokens: int = 50_000_000        # per run: R1, R2 and R3 each read 50M tokens
    batch: int = 32
    lr: float = 1e-3
    min_lr_frac: float = 0.1        # where the decay ends, as a fraction of lr
    warmup_steps: int = 200
    decay_frac: float = 0.2         # R2 and R3 decay over their last 20%; R1 never decays
    weight_decay: float = 0.1
    beta2: float = 0.95
    grad_clip: float = 1.0
    eval_every_tokens: int = 2_500_000
    eval_batches: int = 20          # 20 x 32 sequences of 512 = 327k validation tokens
    log_every: int = 25
    loss_chunk: int = 8192          # tokens per slice of the chunked loss
    ckpt_minutes: float = 8.0

@dataclass(frozen=True)
class DataCfg:
    train_tokens: int = 100_500_000  # R1's 50M, then the continuations' 50M, plus a margin
    val_tokens: int = 1_000_000
    tokenizer_chars: int = 40_000_000
    fineweb_name: str = "sample-10BT"

MC, TC, DC = ModelCfg(), TrainCfg(), DataCfg()
if PROFILE == "cpu":
    MC = replace(MC, vocab=1024, d=64, n_layer=4, n_head=4, seq=128)
    TC = replace(TC, tokens=300_000, batch=8, warmup_steps=20, eval_every_tokens=75_000,
                 eval_batches=4, log_every=10, loss_chunk=512)
    DC = replace(DC, train_tokens=650_000, val_tokens=40_000, tokenizer_chars=1_000_000)
elif PROFILE == "smoke":
    TC = replace(TC, tokens=2_000_000, warmup_steps=20, eval_every_tokens=500_000, eval_batches=5)
TC = replace(TC, ckpt_minutes=float(os.environ.get("ERA5_CKPT_MIN", TC.ckpt_minutes)))

STEPS = TC.tokens // (TC.batch * MC.seq)          # optimiser steps per run

def param_breakdown(mc=MC):
    """Where the dense model's weights live (the output layer shares the embedding)."""
    emb = mc.vocab * mc.d + mc.seq * mc.d
    attn = mc.n_layer * 4 * mc.d * mc.d
    ffn = mc.n_layer * 2 * mc.d * (mc.mlp_mult * mc.d)
    norms = mc.n_layer * 4 * mc.d + 2 * mc.d
    return {"embeddings": emb, "attention": attn, "feed-forward": ffn, "layer norms": norms,
            "total": emb + attn + ffn + norms}

PB = param_breakdown()
print(f"model   : {MC.n_layer} layers x width {MC.d}, {MC.n_head} heads, FFN {MC.mlp_mult*MC.d} neurons, "
      f"vocab {MC.vocab:,}, sequence {MC.seq}")
for k, v in PB.items():
    print(f"          {k:<13} {v/1e6:6.2f}M" + ("" if k == "total" else f"  ({v/PB['total']*100:4.1f}%)"))
print(f"training: {TC.tokens/1e6:.1f}M tokens per run, batch {TC.batch} x {MC.seq} "
      f"-> {STEPS:,} steps; lr {TC.lr}, warm-up {TC.warmup_steps} steps, "
      f"decay over the last {TC.decay_frac:.0%} of R2/R3 only")
R["config"] = {"model": asdict(MC), "train": asdict(TC), "data": asdict(DC), "params": PB, "steps": STEPS}

model   : 16 layers x width 320, 5 heads, FFN 1280 neurons, vocab 8,192, sequence 512
          embeddings      2.79M  (12.4%)
          attention       6.55M  (29.2%)
          feed-forward   13.11M  (58.3%)
          layer norms     0.02M  ( 0.1%)
          total          22.47M
training: 50.0M tokens per run, batch 32 x 512 -> 3,051 steps; lr 0.001, warm-up 200 steps, decay over the last 20% of R2/R3 only


## Cell 3 — storage and caching

### What this block does
Mounts Google Drive and sets up two folders. **outputs/** holds what you download: logs,
plots, `summary.json`. **cache/** holds machinery: the tokenised data, finished runs,
mid-run checkpoints and R1's final checkpoint, which v2 and v3 start from.

### How it works
A finished run is saved together with a hash of its settings **and of the code that produced
it**. Change a setting or edit a function, and the old result is recomputed instead of
silently replayed. The version number is deliberately *not* part of the hash, so moving from
v1 to v2 does not retrain R1.

In [3]:
RESET_CACHE = False        # True forces every run to recompute

try:
    from google.colab import drive as _colab_drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DRIVE_OK = False
if IN_COLAB:
    try:
        _colab_drive.mount("/content/drive")
        ROOT, DRIVE_OK = "/content/drive/MyDrive/era5_session14", True
    except Exception as e:
        ROOT = "/content/era5_session14"
        print("!" * 74)
        print(f"!! DRIVE MOUNT FAILED ({type(e).__name__}). Falling back to {ROOT}, which")
        print("!! dies with the runtime. A disconnect will lose finished runs, including R1.")
        print("!" * 74)
    OUT_DIR, CACHE_DIR = os.path.join(ROOT, "outputs"), os.path.join(ROOT, "cache")
else:
    ROOT, OUT_DIR, CACHE_DIR = ".", "outputs", "checkpoints"
DATA_DIR = os.path.join(CACHE_DIR, "data_" + hashlib.sha1(
    json.dumps({"d": asdict(DC), "v": MC.vocab}, sort_keys=True).encode()).hexdigest()[:8])
if PROFILE != "full":                     # smoke and CPU results never mix with the real ones
    OUT_DIR = os.path.join(OUT_DIR, PROFILE)
    CACHE_DIR = os.path.join(CACHE_DIR, PROFILE)
LOG_DIR = os.path.join(OUT_DIR, "logs")
for _d in (OUT_DIR, CACHE_DIR, DATA_DIR, LOG_DIR):
    os.makedirs(_d, exist_ok=True)

def out(name):   return os.path.join(OUT_DIR, name)
def cache(name): return os.path.join(CACHE_DIR, name)

def _h(obj):
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:10]

BASE_KEY = _h({"model": asdict(MC), "train": {k: v for k, v in asdict(TC).items() if k != "ckpt_minutes"},
               "data": asdict(DC), "seed": SEED, "device": DEVICE, "profile": PROFILE})

def _src_hash(*objs):
    """Hash of the SOURCE of the functions and classes a cached result depends on."""
    parts = []
    for o in objs:
        try:
            parts.append(inspect.getsource(o))
        except (OSError, TypeError):
            parts.append("")
    return hashlib.sha1("".join(parts).encode()).hexdigest()[:10]

def _atomic_save(obj, path, as_json=False):
    tmp = path + ".tmp"
    if as_json:
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(obj, f, indent=1, default=float)
    else:
        torch.save(obj, tmp)
    os.replace(tmp, path)

print(f"in colab   : {IN_COLAB}  (drive mounted: {DRIVE_OK})")
print(f"outputs    : {OUT_DIR}   <- download this folder")
print(f"cache      : {CACHE_DIR}   <- R1's checkpoint lives here; keep it for v2 and v3")
print(f"base key   : {BASE_KEY}")

Mounted at /content/drive
in colab   : True  (drive mounted: True)
outputs    : /content/drive/MyDrive/era5_session14/outputs   <- download this folder
cache      : /content/drive/MyDrive/era5_session14/cache   <- R1's checkpoint lives here; keep it for v2 and v3
base key   : e584d66077


## Cell 4 — data: FineWeb, a tokenizer of our own, tokenised once

### What this block does
Streams FineWeb (sample-10BT) from Hugging Face, trains an 8,192-token byte-pair tokenizer on
the start of it, and turns the text into two flat files of token ids: 100.5M training tokens
and 1M validation tokens from **separate documents**. Everything is saved to Drive, so this
runs once (it is the slowest part of v1, roughly 10–20 minutes).

### How the runs split the data
The training file is cut into windows of 513 tokens (512 inputs plus the next token as the
target), and their order is shuffled once with a fixed seed. R1 reads the first 97,632
windows of that order (50M tokens). R2 and R3 both read the next 97,632, in the same order, so
the control and the MoE see exactly the same new text.

### What you should see
The token counts, a sample sentence round-tripped through the tokenizer, and about 3.7
characters per token (session 13 measured 3.73).

In [4]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

EOT = "<|eot|>"

def _fineweb_texts():
    from datasets import load_dataset
    ds = load_dataset("HuggingFaceFW/fineweb", name=DC.fineweb_name, split="train",
                      streaming=True)
    for row in ds:
        yield row["text"]

def _fallback_texts():
    """Only used by the CPU profile when the FineWeb stream is unreachable."""
    import urllib.request
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    path = os.path.join(DATA_DIR, "tinyshakespeare.txt")
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    text = open(path, encoding="utf-8").read()
    for _ in range(40):                      # small corpus: repeat it (CPU profile only)
        for block in text.split("\n\n"):
            if len(block) > 64:
                yield block

def build_data():
    meta_path = os.path.join(DATA_DIR, "meta.json")
    want = {"data": asdict(DC), "vocab": MC.vocab}
    if os.path.exists(meta_path):
        meta = json.load(open(meta_path, encoding="utf-8"))
        if meta.get("want") == want:
            print(f"[cached] tokenised data found in {DATA_DIR}")
            return meta
    t0 = time.time()
    source = "fineweb"
    try:
        stream = _fineweb_texts()
        first = next(stream)
    except Exception as e:
        if PROFILE != "cpu":
            raise
        print(f"FineWeb unreachable ({type(e).__name__}); CPU profile falls back to Tiny Shakespeare")
        source, stream = "tinyshakespeare", _fallback_texts()
        first = next(stream)

    def docs():
        yield first
        yield from stream

    it = docs()
    # 1) validation documents first, kept aside as raw text (6 chars per token is generous)
    val_texts, n = [], 0
    for t in it:
        val_texts.append(t); n += len(t)
        if n >= DC.val_tokens * 6:
            break
    # 2) the tokenizer is trained on the next documents, which also start the training set
    tok_texts, n = [], 0
    for t in it:
        tok_texts.append(t); n += len(t)
        if n >= DC.tokenizer_chars:
            break
    print(f"source {source}: {len(val_texts):,} validation docs, {len(tok_texts):,} docs "
          f"({sum(len(x) for x in tok_texts)/1e6:.1f}M chars) for the tokenizer")

    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=MC.vocab, special_tokens=[EOT], min_frequency=2,
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tok.train_from_iterator(tok_texts, trainer=trainer)
    tok.save(os.path.join(DATA_DIR, "tokenizer.json"))
    eot = tok.token_to_id(EOT)
    print(f"tokenizer trained: vocabulary {tok.get_vocab_size():,} in {time.time()-t0:.0f}s")

    def encode_to(texts_iter, limit, path):
        chunks, n, chars, batch = [], 0, 0, []
        def flush(batch):
            nonlocal n, chars
            enc = tok.encode_batch(batch)
            for e, t in zip(enc, batch):
                ids = np.asarray(e.ids + [eot], dtype=np.uint16)
                chunks.append(ids); n += len(ids); chars += len(t)
        for t in texts_iter:
            batch.append(t)
            if len(batch) == 512:
                flush(batch); batch = []
                if n >= limit:
                    break
        if batch and n < limit:
            flush(batch)
        arr = np.concatenate(chunks)[:limit]
        if len(arr) < limit:
            raise RuntimeError(f"only {len(arr):,} tokens available for {path}, wanted {limit:,}")
        arr.tofile(path)
        return len(arr), chars / max(n, 1)

    n_val, _ = encode_to(iter(val_texts), DC.val_tokens, os.path.join(DATA_DIR, "val.bin"))
    def train_iter():
        yield from tok_texts
        yield from it
    n_tr, cpt = encode_to(train_iter(), DC.train_tokens, os.path.join(DATA_DIR, "train.bin"))
    meta = {"want": want, "source": source, "train_tokens": n_tr, "val_tokens": n_val,
            "chars_per_token": cpt, "eot_id": eot, "vocab_size": tok.get_vocab_size(),
            "seconds": time.time() - t0}
    _atomic_save(meta, meta_path, as_json=True)
    print(f"data prepared in {meta['seconds']/60:.1f} min")
    return meta

DATA_META = build_data()
TOK = Tokenizer.from_file(os.path.join(DATA_DIR, "tokenizer.json"))
TRAIN = np.memmap(os.path.join(DATA_DIR, "train.bin"), dtype=np.uint16, mode="r")
VAL = np.memmap(os.path.join(DATA_DIR, "val.bin"), dtype=np.uint16, mode="r")
assert DATA_META["vocab_size"] <= MC.vocab

class Windows:
    """Fixed windows of seq+1 tokens in a fixed shuffled order. Window k is always the same."""
    def __init__(self, data, seq, shuffle, seed=SEED):
        self.data, self.seq = data, seq
        n = (len(data) - 1) // seq
        self.order = np.random.default_rng(seed).permutation(n) if shuffle else np.arange(n)
    def __len__(self):
        return len(self.order)
    def get(self, start, count, device=DEVICE):
        idx = self.order[start:start + count]
        x = np.stack([self.data[i * self.seq: i * self.seq + self.seq + 1] for i in idx]).astype(np.int64)
        x = torch.from_numpy(x)
        if device == "cuda":
            x = x.pin_memory().to(device, non_blocking=True)
        return x[:, :-1], x[:, 1:]

TRAIN_W = Windows(TRAIN, MC.seq, shuffle=True)
VAL_W = Windows(VAL, MC.seq, shuffle=False)
R1_WINDOWS = STEPS * TC.batch                    # R1 reads windows [0, R1_WINDOWS)
CONT_START = R1_WINDOWS                          # R2 and R3 read [R1_WINDOWS, 2 x R1_WINDOWS)
assert 2 * R1_WINDOWS <= len(TRAIN_W), "not enough training windows for R1 plus the continuations"
_sample = "A mixture of experts runs only a few small networks for each token."
_enc = TOK.encode(_sample)
print(f"source           : {DATA_META['source']}")
print(f"training tokens  : {len(TRAIN):,}  ({len(TRAIN_W):,} windows of {MC.seq}+1)")
print(f"  R1 reads       : windows 0 .. {R1_WINDOWS-1:,}  ({R1_WINDOWS*MC.seq/1e6:.1f}M tokens)")
print(f"  R2, R3 read    : windows {CONT_START:,} .. {2*R1_WINDOWS-1:,}  (the next {R1_WINDOWS*MC.seq/1e6:.1f}M)")
print(f"validation tokens: {len(VAL):,}  (separate documents)")
print(f"chars per token  : {DATA_META['chars_per_token']:.2f}")
print(f"round trip       : {len(_enc.ids)} tokens -> {TOK.decode(_enc.ids)!r}")
R["data"] = {k: DATA_META[k] for k in ("source", "train_tokens", "val_tokens", "chars_per_token", "vocab_size")}

README.md:   0%|          | 0.00/44.3k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

source fineweb: 1,913 validation docs, 13,017 docs (40.0M chars) for the tokenizer
tokenizer trained: vocabulary 8,192 in 37s
data prepared in 4.6 min
source           : fineweb
training tokens  : 100,500,000  (196,289 windows of 512+1)
  R1 reads       : windows 0 .. 97,631  (50.0M tokens)
  R2, R3 read    : windows 97,632 .. 195,263  (the next 50.0M)
validation tokens: 1,000,000  (separate documents)
chars per token  : 3.73
round trip       : 17 tokens -> 'A mixture of experts runs only a few small networks for each token.'


## Cell 5 — the dense model  (frozen after v1)

### What this block does
A small GPT: token and position embeddings, 16 pre-norm blocks, a final norm, and an output
layer that shares the embedding matrix. Each block is attention followed by the
**feed-forward network (FFN)**: `up` widens 320 → 1,280, GELU, `down` narrows 1,280 → 320.

### Why the FFN is written this way
The FFN is the part v2 turns into experts. It is kept as its own module, `block.mlp`, with
two plain matrices, so v2 can swap it for an MoE layer **without changing any code in this
cell**. That matters: the cache recognises R1 by the source of this cell, and editing it
would retrain R1.

The loss is computed in slices of 8,192 tokens and recomputed in the backward pass, as in
session 13, so the full table of scores over the vocabulary never sits in memory at once.

In [5]:
class Attention(nn.Module):
    def __init__(self, mc):
        super().__init__()
        self.n_head = mc.n_head
        self.qkv = nn.Linear(mc.d, 3 * mc.d, bias=False)
        self.proj = nn.Linear(mc.d, mc.d, bias=False)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class MLP(nn.Module):
    def __init__(self, mc):
        super().__init__()
        self.up = nn.Linear(mc.d, mc.mlp_mult * mc.d, bias=False)
        self.down = nn.Linear(mc.mlp_mult * mc.d, mc.d, bias=False)
    def forward(self, x):
        return self.down(F.gelu(self.up(x)))

class Block(nn.Module):
    def __init__(self, mc):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(mc.d), Attention(mc)
        self.ln2, self.mlp = nn.LayerNorm(mc.d), MLP(mc)
    def forward(self, x):
        x = x + self.attn(self.ln1(x)).to(x.dtype)
        return x + self.mlp(self.ln2(x)).to(x.dtype)

def _ce_slice(h, w, y):
    logits = h @ w.t()
    if logits.dtype in (torch.float16, torch.bfloat16):
        logits = logits.float()
    return F.cross_entropy(logits, y, reduction="sum")

def chunked_ce(h, w, y, chunk):
    """Mean cross-entropy, computed slice by slice and recomputed in the backward pass."""
    h, y = h.reshape(-1, h.size(-1)), y.reshape(-1)
    total = h.new_zeros((), dtype=torch.float64 if h.dtype == torch.float64 else torch.float32)
    for i in range(0, h.size(0), chunk):
        if torch.is_grad_enabled():
            total = total + checkpoint(_ce_slice, h[i:i + chunk], w, y[i:i + chunk], use_reentrant=False)
        else:
            total = total + _ce_slice(h[i:i + chunk], w, y[i:i + chunk])
    return total / y.numel()

class GPT(nn.Module):
    def __init__(self, mc):
        super().__init__()
        self.mc = mc
        self.wte = nn.Embedding(mc.vocab, mc.d)
        self.wpe = nn.Embedding(mc.seq, mc.d)
        self.blocks = nn.ModuleList(Block(mc) for _ in range(mc.n_layer))
        self.ln_f = nn.LayerNorm(mc.d)
        self.apply(self._init)
        for b in self.blocks:        # GPT-2 style: shrink the layers that write into the stream
            for w in (b.attn.proj.weight, b.mlp.down.weight):
                nn.init.normal_(w, std=0.02 / math.sqrt(2 * mc.n_layer))

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def hidden(self, idx):
        T = idx.size(1)
        x = self.wte(idx) + self.wpe(torch.arange(T, device=idx.device))
        for blk in self.blocks:
            x = blk(x)
        return self.ln_f(x)

    def forward(self, idx, targets, chunk=None):
        return chunked_ce(self.hidden(idx), self.wte.weight, targets, chunk or TC.loss_chunk)

    @torch.no_grad()
    def logits(self, idx):
        return self.hidden(idx) @ self.wte.weight.t()

def make_dense():
    torch.manual_seed(SEED)
    return GPT(MC).to(DEVICE)

_m = make_dense()
_n = sum(p.numel() for p in _m.parameters())
assert _n == PB["total"], (_n, PB["total"])
print(f"parameters: {_n:,}  (matches the breakdown in Cell 2)")
del _m

parameters: 22,467,200  (matches the breakdown in Cell 2)


## Cell 6 — the training loop  (frozen after v1)

### What this block does
One function trains any model for a given number of steps and returns its history. R1 uses
it now; R2 and R3 will use the same function in v3, which is why it takes a few arguments
that R1 does not need yet:

| argument | what R1 uses | what v3 will use |
|---|---|---|
| `init` | nothing: random weights | R1's final weights (and optimiser state) |
| `window_start` | 0 | the first window after R1's data |
| `warmup`, `decay` | 200 steps up, no decay | no warm-up (already warm), decay over the last 20% |
| `extra_loss`, `after_step` | not used | the MoE's small balancing loss and bias update |

### How it works
- AdamW, weight decay 0.1 on the weight matrices only, gradient clipping at 1.0.
- fp16 autocast with loss scaling on the GPU.
- Validation loss on the same 327k held-out tokens every 2.5M training tokens.
- **Speed** is measured on the GPU clock over training steps only (the first 10 steps and
  every validation pass are excluded). **Peak memory** is `torch.cuda.max_memory_allocated()`.
- A checkpoint is saved every 8 minutes; after a disconnect the run resumes from it.
- At the end, the final weights, optimiser moments and loss scaler are saved as the run's
  **final checkpoint**. R1's final checkpoint is the starting point for everything in v2 and v3.

In [6]:
def make_opt(model, lr):
    decay = [p for n, p in model.named_parameters() if p.dim() >= 2]
    no_decay = [p for n, p in model.named_parameters() if p.dim() < 2]
    groups = [{"params": decay, "weight_decay": TC.weight_decay},
              {"params": no_decay, "weight_decay": 0.0}]
    kw = {"fused": True} if DEVICE == "cuda" else {}
    return torch.optim.AdamW(groups, lr=lr, betas=(0.9, TC.beta2), eps=1e-8, **kw)

def lr_at(step, steps, lr, warmup, decay):
    """Warmup-stable-decay: linear up over `warmup` steps, flat, then linear down to
    min_lr_frac x lr over the last `decay` steps (decay = 0 means it never comes down)."""
    if step < warmup:
        return lr * (step + 1) / warmup
    if decay and step >= steps - decay:
        t = (step - (steps - decay) + 1) / decay
        return lr * (1 - (1 - TC.min_lr_frac) * t)
    return lr

def _sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

@torch.no_grad()
def evaluate(model):
    model.eval()
    ctx = torch.autocast("cuda", dtype=AMP_DTYPE) if AMP else nullcontext()
    tot = 0.0
    for i in range(TC.eval_batches):
        x, y = VAL_W.get(i * TC.batch, TC.batch)
        with ctx:
            tot += model(x, y).item()
    model.train()
    return tot / TC.eval_batches

def _ckpt_path(key):  return cache(f"ckpt_{BASE_KEY}_{key}.pt")
def _final_path(key): return cache(f"final_{BASE_KEY}_{key}.pt")
def _run_path(key):   return cache(f"run_{BASE_KEY}_{key}.json")

def train_run(key, make_model, steps, warmup, decay, window_start=0, init=None,
              extra_loss=None, after_step=None, lr=None):
    batch, lr = TC.batch, lr or TC.lr
    if window_start + steps * batch > len(TRAIN_W):
        raise ValueError(f"{key}: needs windows up to {window_start + steps*batch:,}, only {len(TRAIN_W):,} exist")
    model = make_model()
    opt = make_opt(model, lr)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    if init is not None:
        model.load_state_dict(init["model"])
        if init.get("opt") is not None:
            opt.load_state_dict(init["opt"])
        if init.get("scaler") is not None and AMP:
            scaler.load_state_dict(init["scaler"])
    ctx = torch.autocast("cuda", dtype=AMP_DTYPE) if AMP else nullcontext()
    eval_every = max(1, TC.eval_every_tokens // (batch * MC.seq))
    hist = {"key": key, "steps": steps, "batch": batch, "lr": lr, "warmup": warmup, "decay": decay,
            "window_start": window_start, "tokens_per_step": batch * MC.seq,
            "train": [], "val": [], "lr_trace": [], "train_seconds": 0.0, "timed_tokens": 0, "extra": {}}
    start = 0
    ck = _ckpt_path(key)
    if not RESET_CACHE and os.path.exists(ck):
        try:
            blob = torch.load(ck, map_location=DEVICE, weights_only=False)
            model.load_state_dict(blob["model"]); opt.load_state_dict(blob["opt"])
            scaler.load_state_dict(blob["scaler"]); hist = blob["hist"]; start = blob["step"] + 1
            print(f"    [resume] '{key}' from step {start:,} of {steps:,}")
        except Exception as e:
            print(f"    [resume failed] '{key}': {type(e).__name__}, starting over")
    if start == 0:                               # loss before any step: where this run begins
        hist["val"].append({"step": 0, "tokens": 0, "loss": evaluate(model), "train_seconds": 0.0})
        print(f"    {key:<14} step      0/{steps:,}  val {hist['val'][0]['loss']:.4f}  (before training)")
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    model.train()
    last_ck, win_t0, win_tok, run_loss = time.time(), None, 0, []
    for step in range(start, steps):
        cur_lr = lr_at(step, steps, lr, warmup, decay)
        for g in opt.param_groups:
            g["lr"] = cur_lr
        x, y = TRAIN_W.get(window_start + step * batch, batch)
        if step == start + 10:
            _sync(); win_t0, win_tok = time.time(), 0
        with ctx:
            loss = model(x, y)
            total = loss + extra_loss(model) if extra_loss else loss
        scaler.scale(total).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), TC.grad_clip)
        scaler.step(opt); scaler.update()
        opt.zero_grad(set_to_none=True)
        if after_step:
            after_step(model, step, hist)
        run_loss.append(loss.item())
        if win_t0 is not None:
            win_tok += batch * MC.seq
        last = step == steps - 1
        is_eval = (step + 1) % eval_every == 0 or last
        if ((step + 1) % TC.log_every == 0 or is_eval) and run_loss:
            hist["train"].append({"step": step + 1, "tokens": (step + 1) * batch * MC.seq,
                                  "loss": float(np.mean(run_loss))})
            hist["lr_trace"].append({"step": step + 1, "lr": cur_lr})
            run_loss = []
        if is_eval:
            if win_t0 is not None:
                _sync(); hist["train_seconds"] += time.time() - win_t0; hist["timed_tokens"] += win_tok
            v = evaluate(model)
            hist["val"].append({"step": step + 1, "tokens": (step + 1) * batch * MC.seq, "loss": v,
                                "train_seconds": hist["train_seconds"]})
            tps = hist["timed_tokens"] / hist["train_seconds"] if hist["train_seconds"] else float("nan")
            print(f"    {key:<14} step {step+1:>6,}/{steps:,}  tokens {(step+1)*batch*MC.seq/1e6:5.1f}M  "
                  f"lr {cur_lr:.2e}  train {hist['train'][-1]['loss']:.4f}  val {v:.4f}  {tps:,.0f} tok/s")
            if win_t0 is not None:
                _sync(); win_t0, win_tok = time.time(), 0
        if time.time() - last_ck > TC.ckpt_minutes * 60 and not last:
            if win_t0 is not None:
                _sync(); hist["train_seconds"] += time.time() - win_t0; hist["timed_tokens"] += win_tok
                win_t0, win_tok = time.time(), 0
            _atomic_save({"model": model.state_dict(), "opt": opt.state_dict(),
                          "scaler": scaler.state_dict(), "hist": hist, "step": step}, ck)
            last_ck = time.time()
    hist["tokens_per_second"] = hist["timed_tokens"] / hist["train_seconds"] if hist["train_seconds"] else None
    hist["peak_mem_bytes"] = torch.cuda.max_memory_allocated() if DEVICE == "cuda" else None
    hist["final_val"] = hist["val"][-1]["loss"]
    hist["final_train"] = hist["train"][-1]["loss"]
    _atomic_save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict()},
                 _final_path(key))
    del model, opt
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return hist

def _replay(hist):
    """Print a reloaded run's progress from its saved log."""
    key, tr = hist["key"], {p["step"]: p["loss"] for p in hist["train"]}
    print(f"    [cached run] '{key}' reloaded, not retrained; progress replayed from its saved log:")
    for p in hist["val"]:
        earlier = [s for s in tr if s <= p["step"]]
        t = f"{tr[max(earlier)]:.4f}" if earlier else "  -   "
        print(f"    {key:<14} step {p['step']:>6,}/{hist['steps']:,}  tokens {p['tokens']/1e6:5.1f}M  "
              f"train {t}  val {p['loss']:.4f}  {p['train_seconds']/60:5.1f} train-min")
    tps, pk = hist.get("tokens_per_second"), hist.get("peak_mem_bytes")
    print(f"    {key:<14} measured: " + (f"{tps:,.0f} tokens/s" if tps else "speed not timed")
          + (f", peak memory {pk/2**20:,.0f} MiB" if pk else ""))

DENSE_CODE = (Attention, MLP, Block, GPT, chunked_ce, _ce_slice, make_dense, train_run, lr_at, make_opt, evaluate, Windows)

def cached_run(key, code, **kw):
    """Train once per configuration and code version; a finished run reloads instantly.
    `code` lists the functions and classes whose source defines this run."""
    path = _run_path(key)
    want = {"src": _src_hash(*code), "kw": {k: (v.__name__ if callable(v) else v) for k, v in kw.items()
                                             if k != "init"}}
    want = json.loads(json.dumps(want, default=str))
    if not RESET_CACHE and os.path.exists(path) and os.path.exists(_final_path(key)):
        blob = json.load(open(path, encoding="utf-8"))
        if blob.get("want") == want:
            _replay(blob["hist"])
            return blob["hist"]
        print(f"    [stale] '{key}': settings or code changed since it was trained, training again")
    hist = train_run(key, **kw)
    _atomic_save({"want": want, "hist": hist}, path, as_json=True)
    _atomic_save(hist, os.path.join(LOG_DIR, f"{key}.json"), as_json=True)
    if os.path.exists(_ckpt_path(key)):
        os.remove(_ckpt_path(key))
    return hist

def load_final(key):
    return torch.load(_final_path(key), map_location=DEVICE, weights_only=False)

print(f"eval every {TC.eval_every_tokens/1e6:.1f}M tokens on {TC.eval_batches*TC.batch*MC.seq:,} "
      f"validation tokens; checkpoint every {TC.ckpt_minutes:g} min")

eval every 2.5M tokens on 327,680 validation tokens; checkpoint every 8 min


## Cell 7 — R1: train the dense model on the first 50M tokens

### What this block does
Trains the dense model from random weights: 200 warm-up steps, then a flat learning rate of
1e-3 to the end. There is no decay, on purpose (see Cell 2). The final weights, optimiser state
and loss scaler are saved as R1's final checkpoint.

### What you should see
A validation line every 2.5M tokens: about 9.0 before training (ln 8,192 = 9.01, a model that
knows nothing), dropping fast at first and then more slowly. Session 13's comparable run
ended at 4.11, but its learning rate had decayed to 10%. Without the decay, expect R1 to end a
little higher, around 4.2–4.4. The speed should be close to session 13's 53,000 tokens/s.

In [7]:
RUN_R1 = "R1_dense"
R1 = cached_run(RUN_R1, DENSE_CODE, make_model=make_dense, steps=STEPS,
                warmup=TC.warmup_steps, decay=0, window_start=0)
_first, _last = R1["val"][0]["loss"], R1["final_val"]
print(f"\nR1: validation loss {_first:.4f} before training -> {_last:.4f} after {STEPS:,} steps "
      f"({(_first - _last):.3f} lower)")
if R1.get("tokens_per_second"):
    print(f"    speed {R1['tokens_per_second']:,.0f} tokens/s"
          + (f", peak memory {R1['peak_mem_bytes']/2**20:,.0f} MiB" if R1.get("peak_mem_bytes") else ""))

    R1_dense       step      0/3,051  val 9.0792  (before training)
    R1_dense       step    152/3,051  tokens   2.5M  lr 7.60e-04  train 6.2760  val 6.3321  61,959 tok/s
    R1_dense       step    304/3,051  tokens   5.0M  lr 1.00e-03  train 5.8727  val 5.9121  58,012 tok/s
    R1_dense       step    456/3,051  tokens   7.5M  lr 1.00e-03  train 5.6094  val 5.6331  56,808 tok/s
    R1_dense       step    608/3,051  tokens  10.0M  lr 1.00e-03  train 5.3921  val 5.3828  56,150 tok/s
    R1_dense       step    760/3,051  tokens  12.5M  lr 1.00e-03  train 5.2123  val 5.2143  55,869 tok/s
    R1_dense       step    912/3,051  tokens  14.9M  lr 1.00e-03  train 5.0516  val 5.0525  55,582 tok/s
    R1_dense       step  1,064/3,051  tokens  17.4M  lr 1.00e-03  train 4.9233  val 4.9060  55,422 tok/s
    R1_dense       step  1,216/3,051  tokens  19.9M  lr 1.00e-03  train 4.7511  val 4.7293  55,334 tok/s
    R1_dense       step  1,368/3,051  tokens  22.4M  lr 1.00e-03  train 4.6511  val 4.6043  

## Cell 8 — Gate: R1's checkpoint reloads to exactly the same model

### What this block does
v2 and v3 build everything from R1's final checkpoint. This gate reloads it into a fresh
model and checks that the validation loss comes out **exactly** as it did at the end of
training. If this ever fails, the conversion in v2 would start from the wrong model.

### What you should see
`[PASS]` and a difference of 0 (or a few units in the last decimal place, from GPU kernels).

In [8]:
class GateError(AssertionError):
    pass

def gate(ok, msg):
    print(f"  [{'PASS' if ok else 'FAIL'}] {msg}")
    if not ok:
        raise GateError(msg)

_blob = load_final(RUN_R1)
_m = make_dense(); _m.load_state_dict(_blob["model"])
_v = evaluate(_m)
_diff = abs(_v - R1["final_val"])
gate(_diff < 1e-5, f"R1 checkpoint reloads: validation loss {_v:.6f} vs {R1['final_val']:.6f} "
                   f"at the end of training (difference {_diff:.1e})")
_opt_steps = [s["step"] for s in _blob["opt"]["state"].values()][:1]
gate(len(_blob["opt"]["state"]) > 0, f"optimiser moments saved too (Adam step count {int(_opt_steps[0]) if _opt_steps else 0:,})")
R["gates"]["r1_reload"] = {"val_reloaded": _v, "val_recorded": R1["final_val"], "diff": _diff}

  [PASS] R1 checkpoint reloads: validation loss 4.123029 vs 4.123029 at the end of training (difference 0.0e+00)
  [PASS] optimiser moments saved too (Adam step count 3,051)


## Cell 9 — what R1 learned: its curve, its schedule, and a sample

### What this block does
Draws R1's loss and learning rate against tokens, and lets the model continue a few prompts,
so you can see what "loss 4.3" looks like as text.

### What you should see
The loss falls steeply in the first few million tokens, then keeps falling slowly. The
learning-rate panel shows the short warm-up ramp and the long flat line. The samples should
be grammatical-looking English that wanders off topic: a 22M model on 50M tokens has learned
the local shape of text, not facts.

In [9]:
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False})

def _save(fig, name):
    fig.tight_layout()
    fig.savefig(out(name), bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {name}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot([p["tokens"]/1e6 for p in R1["train"]], [p["loss"] for p in R1["train"]],
           color="#cbc5bb", lw=1, label="train (25-step means)")
ax[0].plot([p["tokens"]/1e6 for p in R1["val"]], [p["loss"] for p in R1["val"]],
           color="#b1521f", marker="o", ms=3, label="validation")
ax[0].set_xlabel("training tokens (millions)"); ax[0].set_ylabel("loss"); ax[0].legend(frameon=False)
ax[0].set_ylim(top=min(10, R1["val"][0]["loss"] + 0.3))
ax[1].plot([p["step"] for p in R1["lr_trace"]], [p["lr"] for p in R1["lr_trace"]], color="#1f6f8b")
ax[1].set_xlabel("step"); ax[1].set_ylabel("learning rate"); ax[1].set_ylim(0, TC.lr * 1.15)
ax[1].set_title("warm-up, then flat: no decay in R1", fontsize=9)
_save(fig, "r1_loss_lr.png")

@torch.no_grad()
def sample(model, prompt, n=40, temperature=0.8, top_k=40, seed=0):
    g = torch.Generator(device="cpu").manual_seed(seed)
    ids = TOK.encode(prompt).ids
    model.eval()
    for _ in range(n):
        x = torch.tensor([ids[-MC.seq:]], device=DEVICE)
        logits = model.logits(x)[0, -1].float().cpu() / temperature
        v, i = torch.topk(logits, top_k)
        nxt = i[torch.multinomial(torch.softmax(v, -1), 1, generator=g)].item()
        ids.append(nxt)
    return TOK.decode(ids)

PROMPTS = ["The best way to learn a new language is", "Scientists have found that",
           "In the middle of the city there is"]
R["samples_r1"] = []
for p in PROMPTS:
    s = sample(_m, p)
    R["samples_r1"].append(s)
    print(f"\n  > {s!r}")
del _m

  saved r1_loss_lr.png

  > 'The best way to learn a new language is to try to make a new generation of people. The first one is to take a new generation to write a new generation of people who have seen the right method of finding and learning to the world.'

  > 'Scientists have found that it was a great day to use the world. It is not an easy method to use. It also comes with a very simple tool that they are very professional and not very important for them. It'

  > 'In the middle of the city there is no lap.\nIn the middle of the Southwest, the middle of the Navy is an indication of the airways, whose main characteristic is the first time to test the class'


## Cell 10 — v1 checks, summary.json, bundle

### What this block does
Collects v1's checks into one list and prints `N/N`, writes `summary.json` and `RESULTS.md`,
and zips `outputs/` into `era5s14_v1_outputs.zip`. Download it together with the executed
notebook and hand both back; v2 is built on what they show.

In [10]:
CHECKS = [
    ("data: 100M tokens, enough windows for R1 and both continuations", 2 * R1_WINDOWS <= len(TRAIN_W)),
    ("R1: finished its step budget", R1["train"][-1]["step"] == STEPS),
    ("R1: validation loss fell", R1["final_val"] < R1["val"][0]["loss"]),
    ("R1: never decayed (last learning rate = peak)", abs(R1["lr_trace"][-1]["lr"] - TC.lr) < 1e-12),
    ("R1: checkpoint reloads to the same validation loss", R["gates"]["r1_reload"]["diff"] < 1e-5),
]
print("V1 CHECKS")
for name, ok in CHECKS:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
_npass = sum(1 for _, ok in CHECKS if ok)
print(f"\nV1: {_npass}/{len(CHECKS)} passed")
R["acceptance"] = {"passed": _npass, "total": len(CHECKS),
                   "checks": [{"name": n, "result": bool(ok)} for n, ok in CHECKS]}
R["runs"] = {"R1": R1}
R["meta"]["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
_atomic_save(R, out("summary.json"), as_json=True)

_tps = R1.get("tokens_per_second")
L_ = [f"# Session 14 results — notebook {NOTEBOOK_VERSION}, profile {PROFILE}\n",
      f"Model {MC.n_layer} × {MC.d}, vocab {MC.vocab:,}, {PB['total']/1e6:.1f}M parameters "
      f"({PB['feed-forward']/PB['total']*100:.0f}% in the feed-forward layers). "
      f"Device {DEVICE}{' (' + GPU_NAME + ')' if GPU_NAME else ''}.\n",
      f"V1 checks: **{_npass}/{len(CHECKS)}**\n",
      "| run | steps | tokens | val before | final train | final val | tokens/s | peak MiB |",
      "|---|---|---|---|---|---|---|---|",
      f"| R1 dense | {STEPS:,} | {STEPS*TC.batch*MC.seq/1e6:.1f}M | {R1['val'][0]['loss']:.4f} | "
      f"{R1['final_train']:.4f} | {R1['final_val']:.4f} | {(_tps or float('nan')):,.0f} | "
      f"{((R1.get('peak_mem_bytes') or float('nan'))/2**20):,.0f} |",
      "\n## Samples from R1\n"] + [f"- {s!r}" for s in R["samples_r1"]] + ["\n## Checks\n"] + \
     [f"- [{'PASS' if c['result'] else 'FAIL'}] {c['name']}" for c in R["acceptance"]["checks"]]
with open(out("RESULTS.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(L_) + "\n")

_zip = out(f"era5s14_{NOTEBOOK_VERSION}_outputs.zip")
with zipfile.ZipFile(_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _dirs, files in os.walk(OUT_DIR):
        for name in files:
            if name.endswith((".png", ".json", ".md")):
                p = os.path.join(root, name)
                z.write(p, arcname=os.path.relpath(p, OUT_DIR))
print(f"\nbundle: {_zip}  ({os.path.getsize(_zip)/1024:.0f} KB)")
print("Download it together with the executed .ipynb.")

V1 CHECKS
  [PASS] data: 100M tokens, enough windows for R1 and both continuations
  [PASS] R1: finished its step budget
  [PASS] R1: validation loss fell
  [PASS] R1: never decayed (last learning rate = peak)
  [PASS] R1: checkpoint reloads to the same validation loss

V1: 5/5 passed

bundle: /content/drive/MyDrive/era5_session14/outputs/era5s14_v1_outputs.zip  (44 KB)
Download it together with the executed .ipynb.
